# Phase 6: Shallow Multi-Layer Perceptron (MLP) Classifier
### Character Classification with Standardized Features & Softmax Confidence
---
### What is Phase 6 and why do we need it?
In Phase 5, we extracted a rich, handcrafted 256-dimensional feature vector ($8 \text{ concentric rings} \times 8 \text{ directions} \times 4 \text{ geometric metrics}$) for every segmented character.

Now in **Phase 6**, we feed these 256 features into our classifier to recognize each handwritten character!

**Strict Architecture Compliance (Zero Deep Learning):**
- **ZERO Deep Learning**: As mandated by the project specification, we do NOT use CNNs, ResNets, RNNs, ViTs, PyTorch, TensorFlow, Keras, Tesseract, or external cloud OCR APIs.
- **Shallow Classical MLP Architecture**:
  * **Input Layer**: 256 neurons (matching the 256-D Circular DDD feature vector)
  * **Hidden Layer 1**: 100 neurons with ReLU activation
  * **Hidden Layer 2**: 90 neurons with ReLU activation
  * **Output Layer**: 47 neurons (corresponding to the 47 classes of EMNIST Balanced)
  * **Classification Head**: Softmax activation producing a normalized probability distribution $\sum_{i=0}^{46} P(C_i) = 1.0$
- **Dataset (EMNIST Balanced)**:
  * 47 total classes: 10 numerical digits (0–9) + 37 balanced Latin characters (merging ambiguous upper/lower pairs like C/c, O/o, S/s).
- **Z-Score Feature Standardization (`StandardScaler`)**:
  * Features are standardized: $z = \frac{x - \mu}{\sigma}$ so every feature contributes equally regardless of unit scale.
- **Mathematical Transparency**:
  * In addition to `scikit-learn`, Phase 6 provides a pure NumPy forward pass ($Z_1 = \text{ReLU}(XW_1 + b_1)$, $Z_2 = \text{ReLU}(Z_1W_2 + b_2)$, $\hat{y} = \text{Softmax}(Z_2W_3 + b_3)$) proving that no black-box dependencies are required for prediction!

In [ ]:
# Step 1: Import required libraries and Phase 6 modules
import sys
import json
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Add Model directory to Python path
module_path = Path("../Model").resolve()
if str(module_path) not in sys.path:
    sys.path.insert(0, str(module_path))

from ingestion import ingest_image
from preprocessing import preprocess_image
from segmentation import segment_document
from normalization import normalize_all_characters
from features import extract_features, extract_features_batch
from predict import (
    load_model,
    load_scaler,
    load_label_mapping,
    predict_character,
    predict_batch,
    forward_pass_numpy,
    MLPInferenceEngine,
)
import config

print("All Phase 6 inference and model modules loaded successfully!")

---
## Step 2: Loading Trained MLP Model, Scaler & Label Mapping
We load our serialized shallow MLP model (`emnist_mlp_model.joblib`), feature scaler (`feature_scaler.joblib`), label mapping (`label_mapping.json`), and training metrics.

In [ ]:
# Load trained model artifacts
model = load_model()
scaler = load_scaler()
label_map = load_label_mapping()

# Load training metrics log
metrics_path = Path("../Model/models/training_metrics.json")
with open(metrics_path, "r", encoding="utf-8") as f:
    metrics = json.load(f)

# Print model parameters and architecture
print("=== Phase 6 Shallow MLP Architecture ===")
print(f"Input Feature Dimensions  : {model.n_features_in_}")
print(f"Hidden Layer Architecture : {model.hidden_layer_sizes}")
print(f"Output Classes (EMNIST)   : {model.n_outputs_}")
print(f"Activation Function       : {model.activation}")
print(f"Training Iterations       : {metrics.get('iterations')}")
print(f"Train Accuracy            : {metrics.get('train_accuracy', 0.0) * 100:.2f}%")
print(f"Validation/Test Accuracy  : {metrics.get('test_accuracy', 0.0) * 100:.2f}%")

# Calculate total trainable parameters
total_params = sum(w.size for w in model.coefs_) + sum(b.size for b in model.intercepts_)
print(f"Total Trainable Parameters: {total_params:,} (Extremely lightweight & transparent!)")

# Plot the loss convergence curve
if "loss_curve" in metrics:
    plt.figure(figsize=(8, 4))
    plt.plot(metrics["loss_curve"], marker="o", markersize=4, color="#1f77b4", linewidth=2)
    plt.title("MLP Training Loss Curve (Cross-Entropy Convergence)", fontweight="bold")
    plt.xlabel("Iteration (Epoch)")
    plt.ylabel("Loss")
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.show()

---
## Step 3: Running End-to-End Inference on Real Document Characters
We run the full pipeline (Phases 1 through 5) on our test photograph to extract normalized characters, extract their 256-D features, and classify them using our Phase 6 `predict_batch` engine.

In [ ]:
# Load test photograph
sample_file = Path("../data/samples/IMG_20261021_212517.jpg")
if not sample_file.exists():
    sample_file = Path("../data/samples/sample_camera_capture_synth.png")

# Run Phases 1 -> 2 -> 3 -> 4 -> 5 -> 6
_, ingested, _ = ingest_image(sample_file)
prep = preprocess_image(ingested)
seg = segment_document(prep.binary_mask, prep.gray_image)
norm_result = normalize_all_characters(seg)

# Run Phase 6 Batch Prediction
engine = MLPInferenceEngine()
predictions = engine.predict_batch(norm_result.characters)

print(f"Total Characters Classified: {len(predictions)}")
avg_confidence = np.mean([p.confidence for p in predictions]) * 100
print(f"Average Classification Confidence: {avg_confidence:.2f}%")

# Preview first 15 predictions
recognized_str = " ".join([p.predicted_char for p in predictions[:15]])
print(f"First 15 Character Predictions: {recognized_str}")

---
## Step 4: Visualizing Individual Character Predictions & Top-3 Probabilities
Let's inspect sample characters: their normalized $16 \times 16$ image, their predicted class, and their top-3 candidate probabilities.

In [ ]:
# Select 6 sample characters to inspect
num_display = min(6, len(norm_result.characters))
fig, axes = plt.subplots(num_display, 2, figsize=(12, 2.5 * num_display))

for i in range(num_display):
    char_obj = norm_result.characters[i]
    pred_obj = predictions[i]
    
    # Left column: Normalized 16x16 image
    ax_img = axes[i, 0] if num_display > 1 else axes[0]
    ax_img.imshow(char_obj.normalized_image, cmap="gray", origin="upper")
    ax_img.set_title(f"Char #{char_obj.char_id}: Predicted '{pred_obj.predicted_char}' ({pred_obj.confidence * 100:.1f}%)", fontweight="bold")
    ax_img.axis("off")
    
    # Right column: Top-3 candidate probabilities horizontal bar chart
    ax_bar = axes[i, 1] if num_display > 1 else axes[1]
    labels = [k[0] for k in pred_obj.top_k]
    confs = [k[1] * 100 for k in pred_obj.top_k]
    colors = ["#2ca02c" if idx == 0 else "#7f7f7f" for idx in range(len(labels))]
    
    y_pos = np.arange(len(labels))
    ax_bar.barh(y_pos, confs, color=colors, height=0.55)
    ax_bar.set_yticks(y_pos)
    ax_bar.set_yticklabels(labels, fontsize=11, fontweight="bold")
    ax_bar.invert_yaxis()  # Highest probability on top
    ax_bar.set_xlim(0, 100)
    ax_bar.set_xlabel("Softmax Probability (%)")
    ax_bar.grid(True, linestyle="--", alpha=0.4, axis="x")

plt.tight_layout()
plt.show()

---
## Step 5: Pure NumPy Forward Pass (Mathematical Transparency)
To demonstrate complete mathematical mastery during vivas or presentations, we can execute inference **without any machine learning framework** using pure NumPy matrix multiplication!

$$\mathbf{z}_1 = \max(0, \mathbf{x} \mathbf{W}_1 + \mathbf{b}_1)$$
$$\mathbf{z}_2 = \max(0, \mathbf{z}_1 \mathbf{W}_2 + \mathbf{b}_2)$$
$$\mathbf{z}_3 = \mathbf{z}_2 \mathbf{W}_3 + \mathbf{b}_3$$
$$P(C_i) = \frac{\exp(z_{3, i})}{\sum_{j} \exp(z_{3, j})}$$

Let's compare scikit-learn's prediction against our hand-coded pure NumPy forward pass.

In [ ]:
# Pick a test character crop
test_char = norm_result.characters[0]
test_feat = extract_features(test_char, mode="circular_ddd")
test_feat_scaled = scaler.transform(test_feat.reshape(1, -1))[0]

# 1. Scikit-Learn prediction
sklearn_probs = model.predict_proba(test_feat_scaled.reshape(1, -1))[0]
sklearn_pred_idx = int(np.argmax(sklearn_probs))
sklearn_pred_char = label_map.get(sklearn_pred_idx, "?")

# 2. Pure NumPy Forward Pass
numpy_probs = forward_pass_numpy(test_feat_scaled, model.coefs_, model.intercepts_)
numpy_pred_idx = int(np.argmax(numpy_probs))
numpy_pred_char = label_map.get(numpy_pred_idx, "?")

# Compare mathematical agreement
max_abs_diff = float(np.max(np.abs(sklearn_probs - numpy_probs)))

print("=== Pure NumPy vs Scikit-Learn Verification ===")
print(f"Scikit-Learn Prediction : '{sklearn_pred_char}' (Class #{sklearn_pred_idx}, Confidence: {sklearn_probs[sklearn_pred_idx]*100:.2f}%)")
print(f"Pure NumPy Prediction   : '{numpy_pred_char}' (Class #{numpy_pred_idx}, Confidence: {numpy_probs[numpy_pred_idx]*100:.2f}%)")
print(f"Maximum Absolute Error  : {max_abs_diff:.10e}")
assert max_abs_diff < 1e-6, "NumPy forward pass does not match scikit-learn!"
print("SUCCESS: Pure NumPy forward pass perfectly replicates the neural network!")

---
## Quick Professor & Viva Cheat Sheet (Phase 6)

> **Q1: Why use a Shallow MLP instead of Deep Learning (CNNs / ResNets / ViTs)?**  
> *Answer:* Our project specification strictly prohibits deep learning. A shallow Multi-Layer Perceptron represents classical, transparent machine learning where geometric features are explicitly engineered rather than learned in an opaque black-box manner.
>
> **Q2: Why the specific architecture 256 -> 100 -> 90 -> 47?**  
> *Answer:* 
> - 256 input nodes match our 256-D Circular DDD feature vector.
> - 100 and 90 hidden nodes form a graceful two-step bottleneck that captures nonlinear feature correlations while preventing overfitting.
> - 47 output nodes correspond to the 47 distinct character classes in the EMNIST Balanced dataset.
>
> **Q3: What is the EMNIST Balanced dataset and why 47 classes?**  
> *Answer:* EMNIST Balanced combines digits (0–9) and Latin letters into 47 classes by merging uppercase and lowercase characters that are identical in shape (such as C/c, O/o, S/s, P/p, V/v, W/w, X/x, Z/z) to prevent classification ambiguity.
>
> **Q4: Why is StandardScaler normalization applied to feature vectors before inference?**  
> *Answer:* Feature metrics have varying scales (e.g., distances vs. densities). Z-score standardization ($z = \frac{x - \mu}{\sigma}$) centers each feature around zero with unit variance, ensuring no single feature dominates gradient updates or activation magnitudes.
>
> **Q5: How is prediction confidence calculated?**  
> *Answer:* Using the Softmax function on the final hidden layer outputs: $P(C_i) = \frac{e^{z_i}}{\sum_{j} e^{z_j}}$. This ensures all outputs are non-negative and sum to exactly 1.0 (a valid probability distribution).
>
> **Q6: Can this network run inference without scikit-learn?**  
> *Answer:* Yes! We demonstrated a pure NumPy forward pass using basic matrix multiplication ($X \cdot W + b$), ReLU non-linearity, and numerically stable Softmax, with absolute error $< 10^{-6}$ compared to scikit-learn.